In [2]:
# -----######-----###### MAIN IMPORTS -----######-----######
import os, re, json, math, unicodedata
import pandas as pd
import requests
from bs4 import BeautifulSoup
from datetime import datetime

# ---------------------------------
# TQM bar
# ---------------------------------
def _tqm_bar(step, total, label):
    width = 28
    frac = step / float(total if total else 1)
    filled = int(width * frac)
    bar = "█" * filled + " " * (width - filled)
    print(f"TQM | {label}: {int(frac*100):3d}%|{bar}| {step}/{total}")

# ---------------------------------
# helpers
# ---------------------------------
def _safe(s):
    if s is None: return ""
    s = str(s)
    s = unicodedata.normalize("NFKD", s)
    s = (s.replace(" ", "_").replace("/", "___").replace(",", "_")
           .replace("(", "").replace(")", "").replace("!", "")
           .replace("’", "").replace("'", "").replace("¿", "").replace("¡", "")
           .replace(":", "").replace(";", "").replace("\"", "")
           .replace("*", "").replace("?", "").replace("<", "").replace(">", "")
           .replace("|", "").replace("[", "").replace("]", "").replace("{", "").replace("}", ""))
    s = s.replace("&", "and")
    return s.strip()

def _read_bandcamp_embeds(html):
    tralbum, embed, band, jsonld = {}, {}, {}, {}

    m = re.search(r'var\s+TralbumData\s*=\s*(\{.*?\});', html, re.S)
    if m:
        try: tralbum = json.loads(m.group(1))
        except: pass

    m = re.search(r'var\s+EmbedData\s*=\s*(\{.*?\});', html, re.S)
    if m:
        try: embed = json.loads(m.group(1))
        except: pass

    m = re.search(r'var\s+BandData\s*=\s*(\{.*?\});', html, re.S)
    if m:
        try: band = json.loads(m.group(1))
        except: pass

    try:
        soup = BeautifulSoup(html, "html.parser")
        for tag in soup.find_all("script", {"type": "application/ld+json"}):
            data = json.loads(tag.text.strip())
            pick = None
            if isinstance(data, list):
                for d in data:
                    if isinstance(d, dict) and d.get("@type") in ("MusicAlbum","MusicRecording"):
                        pick = d; break
            elif isinstance(data, dict) and data.get("@type") in ("MusicAlbum","MusicRecording"):
                pick = data
            if pick:
                jsonld = pick
                break
    except:
        pass

    return tralbum, embed, band, jsonld

def _detect_kind_and_album_url(tralbum, html_url):
    kind = "unknown"
    album_url = None

    tinfo = tralbum.get("trackinfo", [])
    if isinstance(tinfo, list) and len(tinfo) >= 2:
        kind = "album"
    elif isinstance(tinfo, list) and len(tinfo) == 1:
        kind = "track"

    if "/album/" in html_url:
        kind = "album"
    elif "/track/" in html_url and kind == "unknown":
        kind = "track"

    if kind == "track":
        album_title_link = tralbum.get("album_title_link") or tralbum.get("album_url")
        if album_title_link:
            if album_title_link.startswith("http"):
                album_url = album_title_link
            else:
                try:
                    base = re.match(r'^(https?://[^/]+)', html_url).group(1)
                    album_url = base + album_title_link
                except:
                    album_url = None
    return kind, album_url

def _kw_filename(title, artist, remix, genre, label, rel_date, pur_date, ext):
    # BPM/KEY are always NA (per your request)
    title = _safe(title)[:25]
    remix = _safe(remix or "original")
    artist = _safe(artist)[:25]
    genre = _safe(genre or "NA")
    label = _safe(label or "NA")
    rel = _safe(rel_date or "NA")
    pur = _safe(pur_date or "NA")

    new_name = (
        f"TRkw_{title}_ARkw_{artist}_MXkw_{remix}_KYkw_NA_"
        f"BPkw_NA_GNkw_{genre}_LBkw_{label}_RYkw_{rel}_PYkw_{pur}{ext}"
    )
    if len(new_name) > 240:
        new_name = new_name[:230] + ext
    return new_name

# ==============================
# -----######-----######  CORE IMPORTABLE FUNCTION  -----######-----######
# _bandcamp_2710_i1_album_GET_df_meta_renames
# ==============================
def _bandcamp_2710_i1_album_GET_df_meta_renames(
    bc_url,
    album_dir,
    audio_extensions,
    custom_artist="",        # optional override; else page value
    custom_genre="",         # optional override
    custom_label="",         # optional override
    custom_release_date="",  # YYYY_MM_DD (optional override)
    custom_purchase_date="", # YYYY_MM_DD (optional override)
    do_rename=False          # set True to actually rename files
):
    """
    From a Bandcamp ALBUM link:
      1) fetch & parse album/track metadata,
      2) PRINT per-song info ("this is the song info I found"),
      3) build TRkw_* proposed names (BPM/KEY = 'NA'),
      4) optionally rename local files in album_dir (mapped by track order).

    Returns: DataFrame with columns:
      Path, file_name, proposed_name, Renamed_Path,
      track_num, title, duration_sec, artist, album, label,
      release_date, genres, bc_track_id, bc_url, parent_album_url (if track URL)
    """
    steps = 6
    _tqm_bar(1, steps, "Fetch page")
    r = requests.get(bc_url, timeout=30)
    r.raise_for_status()
    html = r.text

    _tqm_bar(2, steps, "Parse embeds")
    tralbum, embed, band, jsonld = _read_bandcamp_embeds(html)

    kind, parent_album = _detect_kind_and_album_url(tralbum, bc_url)
    if kind == "track":
        _tqm_bar(3, steps, "Detected TRACK (not album)")
        print(f"⚠️ Passed a track URL. Parent album might be: {parent_album}")
        return pd.DataFrame([{
            "bc_url": bc_url,
            "detected_type": "track",
            "parent_album_url": parent_album
        }])
    elif kind != "album":
        _tqm_bar(3, steps, "Unknown type")
        raise ValueError("Could not confidently detect an album page. Provide an /album/ URL.")

    _tqm_bar(3, steps, "Detected ALBUM")

    # Album-level fields
    # artist
    artist = (custom_artist or
              tralbum.get("artist") or
              embed.get("artist") or
              band.get("name") or
              (jsonld.get("byArtist", {}).get("name") if isinstance(jsonld.get("byArtist", {}), dict) else ""))
    if not artist and isinstance(jsonld.get("byArtist", {}), list) and jsonld["byArtist"]:
        artist = jsonld["byArtist"][0].get("name", "")
    artist = artist or ""

    # album title
    album_title = (tralbum.get("current", {}).get("title") or
                   embed.get("album_title") or
                   jsonld.get("name") or "")

    # label
    label = custom_label or band.get("name") or ""

    # release date
    release_date = "NA"
    if custom_release_date:
        release_date = custom_release_date
    else:
        rel = tralbum.get("album_release_date") or tralbum.get("publish_date") or jsonld.get("datePublished")
        if rel:
            # try a couple of formats
            parsed = None
            try:
                parsed = datetime.strptime(rel[:10], "%Y-%m-%d")
            except:
                try:
                    parsed = datetime.fromisoformat(rel.replace("Z","").split(" ")[0])
                except:
                    parsed = None
            if parsed:
                release_date = parsed.strftime("%Y_%m_%d")

    # genre/tags
    tags = tralbum.get("tags") or []
    if isinstance(tags, list):
        tags = [t for t in tags if isinstance(t, str)]
    genre = custom_genre or (tags[0] if tags else (jsonld.get("genre") if isinstance(jsonld.get("genre"), str) else ""))
    genres_joined = ", ".join(tags) if tags else (jsonld.get("genre") if isinstance(jsonld.get("genre"), str) else "")

    # tracks
    _tqm_bar(4, steps, "Extract tracks")
    tinfo = tralbum.get("trackinfo", []) or []
    tracks = []
    for i, t in enumerate(tinfo, start=1):
        title = t.get("title") or f"Track_{i}"
        dur = t.get("duration") if t.get("duration") is not None else math.nan
        tid = t.get("id") or t.get("track_id")
        remix = "original"
        tl = title.lower()
        if any(k in tl for k in ("remix","mix","edit","version")):
            remix = title
        tracks.append({
            "track_num": i,
            "title": title,
            "duration_sec": dur,
            "bc_track_id": tid,
            "remix": remix
        })

    # local files
    _tqm_bar(5, steps, "Scan local folder")
    if not os.path.isdir(album_dir):
        raise FileNotFoundError(f"Album directory not found: {album_dir}")

    exts = set([e.lower() for e in audio_extensions]) if audio_extensions else None
    local_files = []
    for name in sorted(os.listdir(album_dir)):
        if name.startswith("._") or name.startswith(".DS"):
            continue
        p = os.path.join(album_dir, name)
        if os.path.isfile(p):
            ext = os.path.splitext(name)[1].lower()
            if exts is None or ext in exts:
                local_files.append(p)

    # map by order
    n = min(len(local_files), len(tracks))
    rows = []

    print("\n🔎 THIS IS THE SONG INFO I FOUND (Bandcamp → Proposed):")
    print("─────────────────────────────────────────────────────")
    for idx in range(n):
        path = local_files[idx]
        fname = os.path.basename(path)
        ext = os.path.splitext(fname)[1]
        t = tracks[idx]

        proposed = _kw_filename(
            title=t["title"],
            artist=custom_artist or artist or "",
            remix=t["remix"],
            genre=custom_genre or genre or "",
            label=custom_label or label or "",
            rel_date=release_date,
            pur_date=(custom_purchase_date or "NA"),
            ext=ext
        )

        # per-track print
        print(f"[{t['track_num']:02d}] title='{t['title']}'  dur={t['duration_sec'] if not math.isnan(t['duration_sec']) else 'NA'}s")
        print(f"     → proposed='{proposed}'")
        rows.append({
            "Path": path,
            "file_name": fname,
            "proposed_name": proposed,
            "Renamed_Path": None,
            "track_num": t["track_num"],
            "title": t["title"],
            "duration_sec": t["duration_sec"],
            "artist": custom_artist or artist or "",
            "album": album_title or "",
            "label": custom_label or label or "",
            "release_date": release_date,
            "genres": genres_joined,
            "bc_track_id": t["bc_track_id"],
            "bc_url": bc_url,
            "parent_album_url": None
        })

    # report mismatches
    extra_tracks = len(tracks) - len(local_files)
    extra_files = len(local_files) - len(tracks)
    if extra_tracks > 0:
        print(f"\n⚠️ Folder has fewer files than Bandcamp tracks: missing {extra_tracks} file(s).")
        for t in tracks[len(local_files):]:
            rows.append({
                "Path": None, "file_name": None, "proposed_name": None, "Renamed_Path": None,
                "track_num": t["track_num"], "title": t["title"], "duration_sec": t["duration_sec"],
                "artist": custom_artist or artist or "", "album": album_title or "", "label": custom_label or label or "",
                "release_date": release_date, "genres": genres_joined, "bc_track_id": t["bc_track_id"],
                "bc_url": bc_url, "parent_album_url": None
            })
    if extra_files > 0:
        print(f"\n⚠️ Folder has extra audio files beyond Bandcamp track count: {extra_files} extra file(s).")

    df = pd.DataFrame(rows)
    _tqm_bar(6, steps, "Ready")

    # rename if requested
    if do_rename and not df.empty:
        total = df["Path"].notna().sum()
        done = 0
        print("\n🛠  Renaming files on disk…")
        for i, row in df.iterrows():
            p = row["Path"]; new = row["proposed_name"]
            if not p or not new:
                continue
            new_path = os.path.join(os.path.dirname(p), new)
            try:
                os.rename(p, new_path)
                df.at[i, "Renamed_Path"] = new_path
                done += 1
                _tqm_bar(done, total, "Renaming")
            except Exception as e:
                print(f"❌ Error renaming: {p} -> {new_path} | {e}")

        print("✅ Rename pass done.")
    else:
        print("\n👉 Preview only (no changes made). Set do_rename=True to apply.")

    return df


In [3]:
# Example usage — tweak and run in your environment:

# 1) Define which extensions to include (pass this from outside, per your preference)
audio_extensions = [".aiff", ".wav", ".flac", ".mp3", ".m4a"]

# 2) Inputs
bc_url = "https://dbh-music.bandcamp.com/track/detroit-noir"   # your album link
album_dir =  "/Users/yerik/Downloads/_____12_BC"                # folder on disk

#3) PREVIEW (prints per-track info; no changes)
df_album = _bandcamp_2710_i1_album_GET_df_meta_renames(
    bc_url=bc_url,
    album_dir=album_dir,
    audio_extensions=audio_extensions,
    custom_artist="",           # leave "" to use page artist; or override, e.g., "YerikoDJ"
    custom_genre="",            # e.g., "Detroit_House"
    custom_label="",            # e.g., "CAI"
    custom_release_date="",     # "YYYY_MM_DD" optional override
    custom_purchase_date="",    # "YYYY_MM_DD" optional override
    do_rename=False             # preview only
)
print(df_album.head(20))



TQM | Fetch page:  16%|████                        | 1/6
TQM | Parse embeds:  33%|█████████                   | 2/6
TQM | Detected TRACK (not album):  50%|██████████████              | 3/6
⚠️ Passed a track URL. Parent album might be: None
                                              bc_url detected_type  \
0  https://dbh-music.bandcamp.com/track/detroit-noir         track   

  parent_album_url  
0             None  


In [4]:
#!#!#!#!#! 0_FNS  
import requests
from bs4 import BeautifulSoup
import pandas as pd

def _scrape_yy10_bandcamp_album_GET_df(url: str) -> pd.DataFrame:
    """
    Scrapes a Bandcamp album page and returns a DataFrame with:  
      - album_name  
      - release_date  
      - label (if any, else NaN)  
      - url  
    """
    resp = requests.get(url)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, 'html.parser')
    
    # album name
    album_name_tag = soup.find('h2', class_='trackTitle')
    if album_name_tag:
        album_name = album_name_tag.text.strip()
    else:
        album_name = None
    
    # release date
    # On Bandcamp album pages, looks like: <div class="tralbumData tralbum-credits"> … released September 29, 2025 …
    release_date = None
    credits_div = soup.find('div', class_='tralbumData tralbum-credits')
    if credits_div:
        txt = credits_div.get_text(separator=' ').strip()
        # look for “released” keyword
        idx = txt.lower().find('released')
        if idx != -1:
            release_date = txt[idx+len('released'):].split()[0:4]  # e.g. ['September','29,','2025']
            release_date = ' '.join(release_date).replace(',', '')
    
    # label (if available)
    label = None
    # On Bandcamp pages the label often appears inside a <a> tag with href including “label”
    for a in soup.select('div.tralbumData > a'):
        href = a.get('href', '')
        if '/label/' in href:
            label = a.text.strip()
            break
    
    # Build DataFrame
    df = pd.DataFrame([{
        'album_name': album_name,
        'release_date': release_date,
        'label': label,
        'url': url
    }])
    return df

#!#!#!#! RUNNING STATEMENTS #!#!#!  
# Example usage:
df_album = _scrape_yy10_bandcamp_album_GET_df(bc_url)

print(df_album)


     album_name     release_date label  \
0  Detroit Noir  October 31 2024  None   

                                                 url  
0  https://dbh-music.bandcamp.com/track/detroit-noir  


In [5]:
# -----######-----###### MAIN IMPORTS -----######-----######
import re, json, requests, pandas as pd
from bs4 import BeautifulSoup
from datetime import datetime

# ---------- TQM ----------
def _tqm_bar(step, total, label):
    width = 28
    frac = 0 if total == 0 else step/float(total)
    filled = int(width*frac)
    bar = "█"*filled + " "*(width-filled)
    print(f"TQM | {label}: {int(frac*100):3d}%|{bar}| {step}/{total}")

# ==============================
#  _bandcamp_2710_album_GET_df_tracks
# ==============================
# -----######-----###### CORE IMPORTABLE FUNCTION -----######-----######
def _bandcamp_2710_album_GET_df_tracks(
    urls_or_df,
    url_col=None
):
    """
    Scrape Bandcamp album/single pages and return track-level metadata.

    Parameters
    ----------
    urls_or_df : str | list[str] | pandas.DataFrame
        - A single Bandcamp URL string, or
        - A list of Bandcamp URLs, or
        - A DataFrame; if provided, also pass `url_col` with the column name holding URLs.
    url_col : str or None
        Required only when urls_or_df is a DataFrame. The column that contains Bandcamp URLs.

    Returns
    -------
    pandas.DataFrame
        Columns:
          - url
          - album_name
          - artist
          - label
          - release_date      (YYYY-MM-DD when possible, otherwise raw text)
          - release_year
          - track_number
          - track_title
          - is_remix          (True/False)
          - remixer           (if parsed)
          - mix_type          (if parsed)
    """
    # ---------- helpers ----------
    def _clean_date(dtxt):
        # Try parsing common patterns like "September 29, 2025"
        dtxt = (dtxt or "").strip()
        for fmt in ["%B %d %Y", "%B %d, %Y", "%d %B %Y", "%Y-%m-%d"]:
            try:
                return datetime.strptime(dtxt.replace(",", ""), fmt).strftime("%Y-%m-%d")
            except Exception:
                pass
        return dtxt or None

    def _extract_release_year(dtxt):
        if not dtxt:
            return None
        m = re.search(r"(\d{4})", dtxt)
        return int(m.group(1)) if m else None

    def _parse_mix_info(title):
        """
        Extract remixer and mix type from typical patterns:
          "Track (Remixer Remix)", "Track (Extended Mix)", "Track (VIP)", "Track (XYZ Edit)", "Track (Dub)"
        """
        if not title:
            return False, None, None

        t = title.strip()
        low = t.lower()
        # quick boolean for remix-like words
        remix_markers = ["remix", "refix", "rework", "edit", "dub", "vip", "version", "extended mix",
                         "radio edit", "instrumental", "club mix", "bootleg"]
        is_remix = any(w in low for w in remix_markers)

        remixer, mix_type = None, None
        # look inside parentheses
        paren = re.search(r"\(([^)]{2,})\)$", t)
        if paren:
            inner = paren.group(1).strip()
            # Try specific "(Something Remix)"
            m = re.search(r"^(?P<who>.+?)\s+(?P<mix>(?:re-?mix|remix|refix|rework|edit|dub|vip|version|extended mix|radio edit|instrumental|club mix|bootleg))$",
                          inner, flags=re.IGNORECASE)
            if m:
                remixer = m.group("who").strip()
                mix_type = m.group("mix").title()
            else:
                # Or pure mix type e.g., "(Extended Mix)"
                m2 = re.search(r"^(?P<mix>(?:re-?mix|remix|refix|rework|edit|dub|vip|version|extended mix|radio edit|instrumental|club mix|bootleg))$",
                               inner, flags=re.IGNORECASE)
                if m2:
                    mix_type = m2.group("mix").title()

        # If still nothing, try simple " - XYZ Remix" at end
        if not remixer and not mix_type:
            m3 = re.search(r"[-–]\s*(?P<who>.+?)\s+(?P<mix>Remix)$", t, flags=re.IGNORECASE)
            if m3:
                remixer = m3.group("who").strip()
                mix_type = "Remix"
                is_remix = True

        return bool(is_remix), remixer, mix_type

    def _parse_tralbum_json(soup):
        """
        Bandcamp embeds a JS object 'TralbumData' containing trackinfo, album title, etc.
        """
        for sc in soup.find_all("script"):
            if sc.string and "TralbumData" in sc.string:
                # Extract JSON after 'TralbumData = ' up to trailing ';'
                m = re.search(r"TralbumData\s*=\s*(\{.*?\});", sc.string, flags=re.DOTALL)
                if m:
                    try:
                        return json.loads(m.group(1))
                    except Exception:
                        pass
        return None

    def _guess_label(soup):
        # Try common places for a label link, fallback to None
        # Bandcamp often uses a sidebar "Label" link on label pages; artist pages may not have one.
        # We'll try og:site_name/artist first; if not, return None here; the caller will default to "Self Release".
        # Sometimes a label appears as an anchor containing '/label/'.
        for a in soup.select("a[href*='/label/']"):
            txt = (a.get_text() or "").strip()
            if txt:
                return txt
        return None

    def _get_artist(soup):
        # Common selectors
        cand = [
            "span[itemprop='byArtist'] a",
            ".artist-override",
            "a[href*='bandcamp.com'] .title",  # fallback
        ]
        for sel in cand:
            el = soup.select_one(sel)
            if el:
                txt = (el.get_text() or "").strip()
                if txt:
                    return txt
        # Fallback to meta tag
        m = soup.find("meta", property="og:site_name")
        if m and m.get("content"):
            return m["content"].strip()
        return None

    def _get_album_title(soup, tralbum):
        if tralbum and tralbum.get("current") and tralbum["current"].get("title"):
            return tralbum["current"]["title"].strip()
        el = soup.select_one("h2.trackTitle")
        if el:
            return el.get_text(strip=True)
        # Fallback to page title
        if soup.title and soup.title.string:
            return soup.title.string.strip()
        return None

    def _get_release_text(soup):
        # The release text often appears inside credits div
        credits_div = soup.find("div", class_="tralbumData tralbum-credits")
        if credits_div:
            txt = credits_div.get_text(" ", strip=True)
            # find after 'released'
            m = re.search(r"released\s+([A-Za-z]+\s+\d{1,2},\s+\d{4})", txt, flags=re.IGNORECASE)
            if m:
                return m.group(1)
            # fallback: any Month Day Year
            m2 = re.search(r"([A-Za-z]+\s+\d{1,2},\s+\d{4})", txt)
            if m2:
                return m2.group(1)
        return None

    def _tracks_from_tralbum(tralbum):
        tracks = []
        if tralbum and tralbum.get("trackinfo"):
            for i, t in enumerate(tralbum["trackinfo"], start=1):
                title = t.get("title")
                tracks.append({"track_number": i, "track_title": title})
        return tracks

    def _tracks_from_dom(soup):
        tracks = []
        rows = soup.select("table#track_table tr.track_row_view")
        if rows:
            for i, r in enumerate(rows, start=1):
                tt = r.select_one("span.track-title")
                title = tt.get_text(strip=True) if tt else None
                tracks.append({"track_number": i, "track_title": title})
        else:
            # Sometimes on singles the DOM structure is simpler
            title_el = soup.select_one("h3#track-name")
            if title_el:
                tracks.append({"track_number": 1, "track_title": title_el.get_text(strip=True)})
        return tracks

    def _scrape_one(url):
        r = requests.get(url)
        r.raise_for_status()
        soup = BeautifulSoup(r.text, "html.parser")

        tralbum = _parse_tralbum_json(soup)

        album_name = _get_album_title(soup, tralbum)
        artist = _get_artist(soup)

        rel_txt = _get_release_text(soup)
        release_date = _clean_date(rel_txt)
        release_year = _extract_release_year(release_date or rel_txt)

        label = _guess_label(soup)
        if not label:
            # Some pages include label-like owner in JSON (less consistent)
            if tralbum and tralbum.get("artist"):
                # If 'artist' differs from site/artist, leave as Self Release anyway
                pass
            label = "Self Release"

        # Tracks
        tracks = _tracks_from_tralbum(tralbum)
        if not tracks:
            tracks = _tracks_from_dom(soup)

        # Single detection: one track on page
        is_single = len(tracks) == 1

        rows = []
        for t in tracks:
            tn = t.get("track_number")
            title = t.get("track_title")
            is_remix, remixer, mix_type = _parse_mix_info(title or "")

            # If single, keep same columns; (requirement says: “if there is a single you have to give me
            # the columns remixer and mix type” — we already include them for all, so we’re good.)
            rows.append({
                "url": url,
                "album_name": album_name,
                "artist": artist,
                "label": label,
                "release_date": release_date,
                "release_year": release_year,
                "track_number": tn,
                "track_title": title,
                "is_remix": is_remix,
                "remixer": remixer,
                "mix_type": mix_type
            })
        return rows

    # ---------- dispatch ----------
    if isinstance(urls_or_df, pd.DataFrame):
        urls = urls_or_df[url_col].dropna().astype(str).tolist()
        out_rows = []
        total = len(urls)
        for i, u in enumerate(urls, start=1):
            _tqm_bar(i, total, "Bandcamp scrape")
            try:
                out_rows.extend(_scrape_one(u))
            except Exception as e:
                out_rows.append({
                    "url": u, "album_name": None, "artist": None, "label": "Self Release",
                    "release_date": None, "release_year": None,
                    "track_number": None, "track_title": None,
                    "is_remix": None, "remixer": None, "mix_type": None,
                    "_error": str(e)
                })
        df_tracks = pd.DataFrame(out_rows)
        # Merge back at album/URL level if you want per-row mapping; since tracks are multiple rows per URL,
        # we’ll just return the track-level DF (safer). You can merge/join upstream as needed.
        return df_tracks

    elif isinstance(urls_or_df, str):
        _tqm_bar(1, 1, "Bandcamp scrape")
        return pd.DataFrame(_scrape_one(urls_or_df))

    else:
        # assume iterable of strings
        urls = list(urls_or_df)
        out_rows = []
        total = len(urls)
        for i, u in enumerate(urls, start=1):
            _tqm_bar(i, total, "Bandcamp scrape")
            out_rows.extend(_scrape_one(u))
        return pd.DataFrame(out_rows)


In [10]:
# Example: single URL
df_bc = _bandcamp_2710_album_GET_df_tracks("https://brvss.bandcamp.com/album/editsitos-004")

# Example: list of URLs
# urls = ["https://brvss.bandcamp.com/album/editsitos-004", "..."]
# df_bc = _bandcamp_2710_album_GET_df_tracks(urls)

# Example: DataFrame column
# df_bc = _bandcamp_2710_album_GET_df_tracks(df, url_col="bandcamp_url")


TQM | Bandcamp scrape: 100%|████████████████████████████| 1/1


In [11]:
df_bc 

,url,album_name,artist,label,release_date,release_year,track_number,track_title,is_remix,remixer,mix_type
0,https://brvss.bandcamp.com/album/editsitos-004,editsitos 004,brvss,Self Release,2025-09-29,2025,1,la bebé (brvss jungleton mix),False,None,None
1,https://brvss.bandcamp.com/album/editsitos-004,editsitos 004,brvss,Self Release,2025-09-29,2025,2,mes tas tentando (brvss club tool),False,None,None
2,https://brvss.bandcamp.com/album/editsitos-004,editsitos 004,brvss,Self Release,2025-09-29,2025,3,rather lie (y2kid's eurodance mix),False,None,None
3,https://brvss.bandcamp.com/album/editsitos-004,editsitos 004,brvss,Self Release,2025-09-29,2025,4,soy mexicano (Pearl Jetta remix),True,Pearl Jetta,Remix
